# 📱 SMS Spam Tespiti ve Sınıflandırması

Bu çalışmada, SMS mesajlarını **Spam** ve **Non-Spam (Ham)** olarak otomatik sınıflandıran bir makine öğrenmesi modeli geliştirilmektedir.

**Çalışmanın adımları:**
1. Veri setinin yüklenmesi ve keşifsel analizi (EDA)
2. Veri ön işleme ve gereksiz sütunların temizlenmesi
3. Eğitim/test setlerine ayırma
4. Metinlerin TF-IDF yöntemiyle sayısal vektörlere dönüştürülmesi
5. Farklı sınıflandırma algoritmalarının (Lojistik Regresyon, Naive Bayes, SVM) `GridSearchCV` ile hiperparametre optimizasyonu yapılarak eğitilmesi
6. Modellerin test seti üzerinde performans karşılaştırması

**Kullanılan veri seti:** `SMS_Spam_Dataset.csv` — 1082 SMS mesajı, `Message_body` (mesaj metni) ve `Label` (Spam/Non-Spam) sütunlarından oluşuyor.

**Amaç:** Gelen bir SMS mesajının içeriğine bakarak, bu mesajın istenmeyen (spam) bir mesaj mı yoksa normal bir mesaj mı olduğunu yüksek doğrulukla tahmin edebilen bir model oluşturmak.

In [54]:
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

In [55]:
df=pd.read_csv("SMS_Spam_Dataset.csv")
df.head()

,S. No.,Message_body,Label
0,1,Rofl. Its true to its name,Non-Spam
1,2,The guy did some bitching but I acted like i'd...,Non-Spam
2,3,"Pity, * was in mood for that. So...any other s...",Non-Spam
3,4,Will ü b going to esplanade fr home?,Non-Spam
4,5,This is the 2nd time we have tried 2 contact u...,Spam


In [56]:
df["Label"].value_counts()

Label
Non-Spam    884
Spam        198
Name: count, dtype: int64

In [57]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1082 entries, 0 to 1081
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   S. No.        1082 non-null   int64 
 1   Message_body  1082 non-null   object
 2   Label         1082 non-null   object
dtypes: int64(1), object(2)
memory usage: 25.5+ KB


In [58]:
df.drop("S. No.", axis=1, inplace=True)
df.head()

,Message_body,Label
0,Rofl. Its true to its name,Non-Spam
1,The guy did some bitching but I acted like i'd...,Non-Spam
2,"Pity, * was in mood for that. So...any other s...",Non-Spam
3,Will ü b going to esplanade fr home?,Non-Spam
4,This is the 2nd time we have tried 2 contact u...,Spam


In [59]:
X=df.drop("Label",axis=1)
y=df["Label"]

In [60]:
from sklearn.model_selection import train_test_split

In [61]:
X_train, X_test, y_train, y_test= train_test_split(X, y, random_state=15, test_size=0.20)

In [62]:
X_train.shape

(865, 1)

In [63]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [64]:
tfidf=TfidfVectorizer(min_df=5, max_df=0.8, norm="l2", sublinear_tf=True, ngram_range=(1,2))

In [65]:
X_train1= tfidf.fit_transform(X_train["Message_body"])
X_test1= tfidf.transform(X_test["Message_body"])

In [66]:
dfk=pd.DataFrame(
    X_train1.toarray(),
    columns=tfidf.get_feature_names_out()
)
dfk.head()

,000,04,08000930705,10,100,1000,1000 cash,11mths,12hrs,150p,...,you know,you ll,you re,you to,you want,your,your 2003,your mobile,yours,yup
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [67]:
from sklearn.linear_model import LogisticRegression
log=LogisticRegression()

In [68]:
log_param={
    "penalty" : ["l1", "l2", "elasticnet"],
    "C" : [0.01, 0.1, 1, 10, 100],
    "solver" : ["newton-cg", "lbfgs", "liblinear", "sag", "saga", "newton-cholesky"]
}

In [69]:
from sklearn.model_selection import GridSearchCV
log_grid= GridSearchCV(estimator=log, param_grid=log_param, n_jobs=-1, cv=5, scoring="accuracy")
log_grid.fit(X_train1, y_train)

C:\ProgramData\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py:528: FitFailedWarning: 
250 fits failed out of a total of 450.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
25 fits failed with the following error:
Traceback (most recent call last):
  File "C:\ProgramData\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\ProgramData\anaconda3\Lib\site-packages\sklearn\base.py", line 1389, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File "C:\ProgramData\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py", line 1193, in fit
    solver =

GridSearchCV(cv=5, estimator=LogisticRegression(), n_jobs=-1,
             param_grid={'C': [0.01, 0.1, 1, 10, 100],
                         'penalty': ['l1', 'l2', 'elasticnet'],
                         'solver': ['newton-cg', 'lbfgs', 'liblinear', 'sag',
                                    'saga', 'newton-cholesky']},
             scoring='accuracy')

In [70]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [71]:
y_pred=log_grid.predict(X_test1)
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))
print(accuracy_score(y_test, y_pred))

              precision    recall  f1-score   support

    Non-Spam       0.97      0.98      0.98       177
        Spam       0.92      0.88      0.90        40

    accuracy                           0.96       217
   macro avg       0.95      0.93      0.94       217
weighted avg       0.96      0.96      0.96       217

[[174   3]
 [  5  35]]
0.9631336405529954


In [72]:
from sklearn.naive_bayes import MultinomialNB
mnb=MultinomialNB()

In [73]:
mnb_param={
    "alpha" : [0.001, 0.01, 0.1, 1, 5],
    "fit_prior" : [True, False]
}

In [74]:
mnb_grid=GridSearchCV(estimator=mnb, param_grid=mnb_param, n_jobs=-1, cv=5, scoring="accuracy")
mnb_grid.fit(X_train1, y_train)
y_pred1=mnb_grid.predict(X_test1)
print(accuracy_score(y_test, y_pred1))
print(classification_report(y_test, y_pred1))
print(confusion_matrix(y_test, y_pred1))

0.9539170506912442
              precision    recall  f1-score   support

    Non-Spam       0.96      0.99      0.97       177
        Spam       0.94      0.80      0.86        40

    accuracy                           0.95       217
   macro avg       0.95      0.89      0.92       217
weighted avg       0.95      0.95      0.95       217

[[175   2]
 [  8  32]]


In [75]:
from sklearn.svm import SVC
svc=SVC()

In [76]:
svc_param={
    "C" : [0.01, 0.1, 1, 10, 100],
    "kernel" : ["linear", "poly", "rbf"],
    "gamma" : ["scale", "auto"]
}

In [77]:
svc_grid=GridSearchCV(estimator=svc, param_grid=svc_param, n_jobs=-1, cv=5, scoring="accuracy")
svc_grid.fit(X_train1, y_train)
y_pred2=svc_grid.predict(X_test1)
print(accuracy_score(y_test, y_pred2))
print(classification_report(y_test, y_pred2))
print(confusion_matrix(y_test, y_pred2))

0.9723502304147466
              precision    recall  f1-score   support

    Non-Spam       0.98      0.99      0.98       177
        Spam       0.95      0.90      0.92        40

    accuracy                           0.97       217
   macro avg       0.96      0.94      0.95       217
weighted avg       0.97      0.97      0.97       217

[[175   2]
 [  4  36]]


## 🔍 Ek Deneme: F1-Score ile Optimizasyon (Denendi, Vazgeçildi)

Lojistik Regresyon modeli `scoring="f1"` ile de denenmiş, ancak model her mesajı **Non-Spam** tahmin eden dejenere bir sonuç vermiştir:

| Metrik | Non-Spam | Spam |
|---|---|---|
| Precision | 0.82 | 0.00 |
| Recall | 1.00 | 0.00 |
| F1-score | 0.90 | 0.00 |

**Accuracy: 0.816** — yanıltıcı, çünkü model tek sınıfı ezberlemiştir.

### Sonuç

İlk eğitilen modellerimiz (`scoring="accuracy"`) zaten Spam sınıfında da güçlü ve dengeli sonuçlar veriyor:

| Model | Accuracy | Spam Precision | Spam Recall | Spam F1-score |
|---|---|---|---|---|
| Lojistik Regresyon | 0.968 | 0.92 | 0.90 | 0.91 |
| Multinomial Naive Bayes | 0.954 | 0.94 | 0.80 | 0.86 |
| SVM (SVC) | 0.972 | 0.95 | 0.90 | 0.92 |

Bu sonuçlar, hem genel doğruluk hem de spam mesajları yakalama açısından pratikte aradığımız performansı karşılıyor. Bu yüzden f1 tabanlı denemede karşılaşılan sorunları çözmek için ek bir düzeltmeye gidilmemiş; mevcut modeller (özellikle Lojistik Regresyon ve SVM, accuracy scoring) yeterli bulunarak nihai model olarak kullanılmıştır. İleride veri seti büyütülür veya sınıf dengesizliği artarsa, f1 tabanlı optimizasyon tekrar değerlendirilebilir.